In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from collections import defaultdict
from tqdm import tqdm
import sys
sys.path.append('..')
from utils import ising_data_builder, H5IsingDataset



# Red Neuronal Convolucional (CNN) para Clasificación del Modelo Ising

Este notebook entrena y evalúa una CNN con arquitectura AlexNet adaptada para clasificar las fases del modelo Ising.



## Construcción del Dataset



In [ ]:
# Crear el dataset
data = ising_data_builder('../data/').h5_path
dataset = H5IsingDataset(data)



## División Train/Test



In [ ]:
n = len(dataset)
labels = np.asarray(dataset.y[:]) 

# Train / Test Split estratificado
train_idx, test_idx = train_test_split(
    np.arange(n),
    test_size=0.2,
    stratify=labels,
    random_state=4
)

# Subsets que no copian datos, solo crean vistas por índices
train_set = Subset(dataset, train_idx.tolist())
test_set = Subset(dataset, test_idx.tolist())

# DataLoaders -> genera los batches
train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=0)



## Definición del Modelo CNN



In [ ]:
# CNN con arquitectura AlexNet adaptada para 10x10
class AlexNetIsing(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        
        # Capas convolucionales (adaptadas para entrada 10x10)
        self.features = nn.Sequential(
            # Primera capa convolucional
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),  # 10x10 -> 10x10
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 10x10 -> 5x5
            
            # Segunda capa convolucional
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),  # 5x5 -> 5x5
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 5x5 -> 2x2
            
            # Tercera capa convolucional
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),  # 2x2 -> 2x2
            nn.ReLU(inplace=True),
        )
        
        # Capas fully connected
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(128 * 2 * 2, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        # Asegurar que la entrada tenga la dimensión del canal
        if x.dim() == 3:  # (batch, height, width)
            x = x.unsqueeze(1)  # (batch, 1, height, width)
        
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.classifier(x)
        return x

# Model, loss, optimizer para CNN
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_cnn = AlexNetIsing(num_classes=1).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)



In [ ]:
# Entrenamiento
def train_epoch(model, loader):
    model.train()
    total_loss, total_correct, total = 0, 0, 0

    for X, y in loader:
        X, y = X.to(device), y.float().to(device)

        logits = model(X).squeeze()  # Forward y eliminar dimensión extra [batch, 1] -> [batch]
        loss = loss_fn(logits, y)    # Loss

        optimizer.zero_grad() # Limpiar gradientes
        loss.backward()       # Backward propagation
        optimizer.step()      # Update weights and biases

        total_loss += loss.item() * X.size(0) # Pérdida del batch

        preds = (torch.sigmoid(logits) >= 0.5).long()     # Logits a labels
        total_correct += (preds == y.long()).sum().item() # Contar correctos
        total += X.size(0) 
        
    # Pérdida promedio y accuracy
    return total_loss / total, total_correct / total 

# Evaluación
def eval_epoch(model, loader):
    model.eval()
    total_loss, total_correct, total = 0, 0, 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.float().to(device)

            logits = model(X).squeeze()  # Eliminar dimensión extra [batch, 1] -> [batch]
            loss = loss_fn(logits, y)

            total_loss += loss.item() * X.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).long()
            total_correct += (preds == y.long()).sum().item()
            total += X.size(0)

    return total_loss / total, total_correct / total



In [ ]:
# Loop de entrenamiento
print("Entrenando CNN (AlexNet)...")
for epoch in tqdm(range(1, 6), desc="Epochs"):
    train_loss, train_acc = train_epoch(model_cnn, train_loader)
    test_loss, test_acc = eval_epoch(model_cnn, test_loader)

    print(f"Epoch {epoch} | "
          f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
          f"Test Loss: {test_loss:.4f}, Acc: {test_acc:.4f}")

